# Building AI Applications with Snowflake Cortex
### RAG, Text-to-SQL & Cortex Code — Workshop Notebook

**Before running this notebook**, complete the SQL Worksheet setup steps in the
lab guide to create `AI_WORKSHOP_DB`, its schemas, and `WORKSHOP_WH`.

| Module | Topic | Time |
|--------|-------|------|
| 1 | Your AI Foundation — Cortex Complete & RAG Concepts | 30 min |
| 2 | Production RAG with Cortex Search | 35 min |
| 3 | Text-to-SQL with Sample Data | 15 min |

## Setup — Imports & Session

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.core import Root
import pandas as pd
import time

session = get_active_session()
session.sql('USE DATABASE AI_WORKSHOP_DB').collect()
session.sql('USE SCHEMA RAG_DATA').collect()
session.sql('USE WAREHOUSE WORKSHOP_WH').collect()

def complete(model, prompt):
    escaped = prompt.replace("'", "''")
    return session.sql(
        f"SELECT SNOWFLAKE.CORTEX.COMPLETE('{model}', '{escaped}')"
    ).collect()[0][0]

Complete = complete  # alias used by RAG_App and Text-to-SQL cells

print('Session ready.')
print('  Database :', session.get_current_database())
print('  Schema   :', session.get_current_schema())
print('  Warehouse:', session.get_current_warehouse())

---
## Module 1 — Your AI Foundation

| Question | Data Type | Technique |
|----------|-----------|----------|
| "What does our policy say about refunds?" | Text, documents | **RAG** — retrieve relevant chunks, ground the LLM |
| "What were total sales last quarter?" | Tables | **Text-to-SQL** — translate to SQL, execute, return result |

### Step 1 — Your First LLM Call

`CORTEX.COMPLETE` calls hosted LLMs **inside** your Snowflake account —
no API key, no data leaving your security perimeter.

In [ ]:
model  = 'mistral-7b'
prompt = 'In two sentences, explain the difference between structured and unstructured data.'
print(complete(model, prompt))

In [ ]:
# Try multiple models — compare quality vs latency
for m in ['mistral-7b', 'snowflake-arctic', 'llama3-70b']:
    print(f'--- {m} ---')
    print(complete(m, prompt))
    print()

### Step 2 — The Hallucination Problem

LLMs are trained on public internet data. Ask them something private or recent
and they fabricate a confident-sounding answer.

In [ ]:
response = complete('mistral-7b', 'What is the exact revenue figure for Snowflake in Q4 FY26?')
print(response)
print()
print('The model guesses, hedges, or states something unverifiable — this is hallucination.')

### Step 3 — RAG: Grounding Fixes Hallucination

Instead of relying on training data, we:
1. **Retrieve** relevant facts from our own Snowflake data
2. **Inject** them into the prompt as context
3. **Instruct** the model to answer using only what we provided

We use `SNOWFLAKE_SAMPLE_DATA` — the TPC-H order dataset pre-loaded in every trial account.
This shows how to take **live data already in Snowflake** and instantly ground an LLM with it.

In [ ]:
# Pull live data from the sample database already in your trial account
top_customers = session.sql("""
    SELECT
        c.C_NAME,
        c.C_MKTSEGMENT,
        ROUND(SUM(o.O_TOTALPRICE), 0)       AS TOTAL_SPENT,
        COUNT(DISTINCT o.O_ORDERKEY)         AS NUM_ORDERS
    FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER c
    JOIN SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS   o ON c.C_CUSTKEY = o.O_CUSTKEY
    GROUP BY 1, 2
    ORDER BY TOTAL_SPENT DESC
    LIMIT 5
""").to_pandas()

lines = ['Top 5 customers by total order value (TPC-H sample database):']
for _, r in top_customers.iterrows():
    lines.append(
        f"  - {r['C_NAME']} | Segment: {r['C_MKTSEGMENT']} "
        f"| Revenue: ${r['TOTAL_SPENT']:,.0f} | Orders: {int(r['NUM_ORDERS'])}"
    )
data_context = '\n'.join(lines)

question = 'Who is the top customer by revenue, what segment are they in, and how many orders have they placed?'

grounded_prompt = (
    'Answer using ONLY the data below. Do not use outside knowledge.\n\n'
    f'Data:\n{data_context}\n\n'
    f'Question: {question}\nAnswer:'
)

print('Live data from SNOWFLAKE_SAMPLE_DATA:')
print(data_context)
print()
print('Grounded AI answer:')
print(complete('mistral-7b', grounded_prompt))

In production, the context is not hand-crafted — it is **retrieved automatically**
from a search index over your full data corpus. That is what Module 2 builds.

### Step 4 — Create Your Text Corpus

The cell below loads 15 Snowflake product feature descriptions —
the document library your RAG app will search.

In [ ]:
session.sql("""
CREATE OR REPLACE TABLE AI_WORKSHOP_DB.RAG_DATA.FEATURE_DOCS (
    feature_name VARCHAR,
    content      VARCHAR
)
""").collect()

rows = [
    ('Cortex Complete', 'Snowflake Cortex Complete provides access to industry-leading large language models directly within Snowflake. Using a simple SQL function or Python API, you can prompt LLMs without moving data outside your security perimeter. Supported models include mistral-7b, mistral-large2, llama3-70b, and snowflake-arctic.'),
    ('Cortex Search', 'Cortex Search enables low-latency, high-quality search and retrieval on your Snowflake data. It combines keyword and semantic vector search and is optimized for RAG use cases. You define a search service over a table column and Snowflake handles indexing and embedding automatically.'),
    ('Cortex Analyst', 'Cortex Analyst is a text-to-SQL service that lets business users ask natural-language questions and receive accurate SQL-powered answers. It uses a YAML semantic model to understand schema and business logic. Cortex Analyst is available via REST API and integrated into Snowsight.'),
    ('Dynamic Tables', 'Dynamic Tables let you define query results as a table that is automatically kept up to date. You write transformation logic in SQL and Snowflake manages scheduling and incremental updates based on a target lag. Both full and incremental refresh are supported.'),
    ('Snowpipe', "Snowpipe is Snowflake's continuous ingestion service. It loads data from staged files as soon as they arrive, without a running warehouse, using a serverless compute model. It can be triggered by cloud storage notifications or REST API calls."),
    ('Snowpark', "Snowpark is Snowflake's developer framework for Python, Java, and Scala code that runs directly on Snowflake compute. It supports data pipelines, ML model training, and UDFs without moving data out of Snowflake."),
    ('Snowflake Notebooks', 'Snowflake Notebooks provide an interactive, cell-based development environment in Snowsight. They support Python, SQL, and Markdown cells running on Snowflake compute. Built-in visualizations and ML integrations are included. Notebooks are available on trial accounts.'),
    ('Cortex Code', "Cortex Code is Snowflake's AI-powered coding assistant. It generates, explains, and debugs SQL and Python from natural language. It understands Snowflake schemas, Cortex APIs, and best practices, and is available as a standalone IDE and as Snowflake Copilot within Snowsight on trial accounts."),
    ('Data Sharing', "Snowflake Data Sharing lets organizations share live, governed data with other Snowflake accounts without data movement. Providers publish to the Marketplace or share directly with consumers, who query the data with their own compute at no storage cost to the provider."),
    ('Snowflake Marketplace', 'The Snowflake Marketplace is a data exchange for datasets, data services, and native apps. Consumers access third-party data directly in Snowflake with no ETL. The Marketplace includes free and paid datasets across finance, weather, healthcare, and more.'),
    ('Cortex Search vs Cortex Analyst', 'Cortex Search is for unstructured data like documents and text, returning relevant passages via hybrid search. Cortex Analyst is for structured tables, generating SQL from natural language. Both can be combined to answer questions across all data types.'),
    ('Task Graphs', 'Task Graphs orchestrate multiple SQL or Snowpark procedures in a dependency-based order. Tasks run on a schedule, triggered by a stream, or by a parent task. Task Graphs replace external orchestrators like Airflow for Snowflake-native pipelines.'),
    ('Iceberg Tables', 'Snowflake Iceberg Tables store data in Apache Iceberg format on your own cloud storage while remaining queryable through Snowflake. They support full DML and interoperability with Spark and Trino, with Snowflake governance and security controls.'),
    ('Streamlit in Snowflake', "Streamlit in Snowflake lets you build and deploy interactive data apps directly in your Snowflake account. Apps run on Snowflake compute, query data natively, and are shareable without external hosting. Available on trial accounts."),
    ('Snowflake AI Trust Layer', "Snowflake's AI Trust Layer ensures data processed by Cortex LLMs never leaves your account. LLM inference runs inside Snowflake's security perimeter with no data retained by model providers, supporting compliance with data residency and privacy requirements."),
]

from snowflake.snowpark import Row
session.write_pandas(
    pd.DataFrame(rows, columns=['FEATURE_NAME', 'CONTENT']),
    table_name='FEATURE_DOCS',
    database='AI_WORKSHOP_DB',
    schema='RAG_DATA',
    overwrite=True
)

count = session.sql('SELECT COUNT(*) AS total FROM AI_WORKSHOP_DB.RAG_DATA.FEATURE_DOCS').collect()
print(f'Corpus loaded: {count[0]["TOTAL"]} documents')
session.sql('SELECT feature_name, LEFT(content, 80) AS preview FROM FEATURE_DOCS').show()

---
## Module 2 — Production RAG with Cortex Search

```
FEATURE_DOCS  →  chunk text  →  CHUNKED_DOCS  →  Cortex Search Service  →  RAG_App
```

### Step 1 — Chunk the Text

Real documents are too long for a single LLM prompt. We split them into overlapping chunks.

> **Cortex Code prompt to try:**  
> *"Chunk the content column in FEATURE_DOCS into 1500-character pieces with 200-character overlap
> using SPLIT_TEXT_RECURSIVE_CHARACTER and store the result in a table called CHUNKED_DOCS"*

In [ ]:
def chunk_doc(text: str, chunk_size: int = 1500, overlap: int = 200) -> list:
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks

docs_df = session.sql(
    'SELECT feature_name, content FROM AI_WORKSHOP_DB.RAG_DATA.FEATURE_DOCS'
).to_pandas()

rows = []
for _, row in docs_df.iterrows():
    for chunk in chunk_doc(row['CONTENT']):
        rows.append({'FEATURE_NAME': row['FEATURE_NAME'], 'CHUNK': chunk})

session.write_pandas(
    pd.DataFrame(rows),
    table_name='CHUNKED_DOCS',
    database='AI_WORKSHOP_DB',
    schema='RAG_DATA',
    overwrite=True
)

result = session.sql('SELECT COUNT(*) AS total FROM AI_WORKSHOP_DB.RAG_DATA.CHUNKED_DOCS').collect()
print(f'Chunks created: {result[0]["TOTAL"]}')

### Step 2 — Create the Cortex Search Service

A single DDL statement creates a fully managed hybrid search index.
Snowflake handles embedding, vectorisation, and retrieval automatically.

> **Cortex Code prompt to try:**  
> *"Create a Cortex Search Service called FEATURE_SEARCH_SERVICE on CHUNKED_DOCS.chunk,
> with feature_name as an attribute, using snowflake-arctic-embed-l-v2.0 and a 1-minute target lag"*

In [ ]:
session.sql("""
CREATE OR REPLACE CORTEX SEARCH SERVICE AI_WORKSHOP_DB.RAG_DATA.FEATURE_SEARCH_SERVICE
  ON chunk
  ATTRIBUTES feature_name
  WAREHOUSE       = WORKSHOP_WH
  TARGET_LAG      = '1 minute'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
  AS (
    SELECT chunk, feature_name
    FROM   AI_WORKSHOP_DB.RAG_DATA.CHUNKED_DOCS
  )
""").collect()

print('Service created. Waiting for initial indexing (up to 3 minutes)...')
for attempt in range(18):
    status_df  = session.sql('SHOW CORTEX SEARCH SERVICES IN SCHEMA AI_WORKSHOP_DB.RAG_DATA').to_pandas()
    status_str = ' '.join(status_df.astype(str).values.flatten()).upper()
    if 'ACTIVE' in status_str:
        print('Service is ACTIVE — ready to query.')
        break
    print(f'  Initializing... ({(attempt + 1) * 10}s elapsed)')
    time.sleep(10)
else:
    print('Still initializing. Proceed and re-run queries below if they fail.')

### Step 3 — Build the RAG Application

> **Cortex Code prompt to try:**  
> *"Write a Python class called RAG_App using snowflake.core that queries FEATURE_SEARCH_SERVICE
> in AI_WORKSHOP_DB.RAG_DATA, retrieves the top 3 chunks, and calls Cortex Complete mistral-large2
> with a grounded prompt that instructs the model to answer only from the context"*

In [ ]:
class RAG_App:
    def __init__(self, session, num_results: int = 3):
        self.num_results = num_results
        root             = Root(session)
        self.svc         = (
            root
            .databases['AI_WORKSHOP_DB']
            .schemas['RAG_DATA']
            .cortex_search_services['FEATURE_SEARCH_SERVICE']
        )

    def retrieve_context(self, question: str) -> list:
        resp = self.svc.search(
            query = question,
            limit = self.num_results
        )
        return [r['CHUNK'] for r in resp.results]

    def generate_answer(self, question: str, context: list) -> str:
        ctx_str = '\n\n'.join(context)
        prompt  = (
            'You are a helpful Snowflake product assistant.\n'
            'Answer using ONLY the context below. '
            'If the answer is not there, say "I do not have that information."\n\n'
            f'Context:\n{ctx_str}\n\n'
            f'Question: {question}\nAnswer:'
        )
        return Complete('mistral-large2', prompt)

    def query(self, question: str) -> str:
        context = self.retrieve_context(question)
        return self.generate_answer(question, context)


app = RAG_App(session)
print('RAG app ready.')

### Step 4 — Test Your Application

In [ ]:
test_questions = [
    'What is the difference between Cortex Search and Cortex Analyst?',
    'How does Snowpipe know when new files arrive?',
    'Can I run Streamlit apps on a trial account?',
    'Does my data leave Snowflake when I call an LLM?',
    'What is the Snowflake AI Trust Layer?',
]

for q in test_questions:
    print(f'Q: {q}')
    print(f'A: {app.query(q)}')
    print('-' * 60)

### Step 5 — Evaluate Response Quality

We use `CORTEX.COMPLETE` as an **LLM judge** — the same model scores each response
on three metrics using 0–1 scales:

| Metric | Question |
|--------|----------|
| **Groundedness** | Is the answer supported by the retrieved context? |
| **Context Relevance** | Did Cortex Search return relevant chunks? |
| **Answer Relevance** | Does the answer address the question? |

In [ ]:
import re

def llm_score(prompt: str) -> float:
    result = Complete('mistral-large2', prompt).strip()
    nums   = re.findall(r'1\.0+|0\.\d+|[01]', result)
    return float(nums[0]) if nums else 0.0

def evaluate(question: str) -> dict:
    context = app.retrieve_context(question)
    answer  = app.generate_answer(question, context)
    ctx_str = '\n'.join(context)

    g = llm_score(
        f'Judge how grounded the answer is in the context.\n'
        f'1.0 = fully supported by context. 0.0 = not supported.\n'
        f'Return ONLY a number between 0.0 and 1.0.\n\n'
        f'Context: {ctx_str}\nAnswer: {answer}\nScore:'
    )
    cr = llm_score(
        f'Judge how relevant the context is for answering the question.\n'
        f'1.0 = directly useful. 0.0 = completely unrelated.\n'
        f'Return ONLY a number between 0.0 and 1.0.\n\n'
        f'Question: {question}\nContext: {ctx_str}\nScore:'
    )
    ar = llm_score(
        f'Judge how well the answer addresses the question.\n'
        f'1.0 = fully addresses it. 0.0 = completely irrelevant.\n'
        f'Return ONLY a number between 0.0 and 1.0.\n\n'
        f'Question: {question}\nAnswer: {answer}\nScore:'
    )
    return {'question': question, 'answer': answer,
            'groundedness': g, 'context_relevance': cr, 'answer_relevance': ar}


results = [evaluate(q) for q in test_questions]
eval_df = pd.DataFrame(results)

print(eval_df[['question','groundedness','context_relevance','answer_relevance']]
      .to_string(index=False))
print(f'\nMean — Groundedness: {eval_df["groundedness"].mean():.2f} | '
      f'Context Relevance: {eval_df["context_relevance"].mean():.2f} | '
      f'Answer Relevance: {eval_df["answer_relevance"].mean():.2f}')

---
## Module 3 — Text-to-SQL with Sample Data

`SNOWFLAKE_SAMPLE_DATA` is pre-loaded in every Snowflake trial account.  
We use the **TPC-H** schema — a standard order-management benchmark — as our structured data source.

Everything in this module runs as Python cells using `session.sql()` — no worksheet switching required.

In [ ]:
# Explore the schema
schema_df = session.sql("""
    SELECT   table_name,
             TO_CHAR(row_count, '999,999,999') AS row_count
    FROM     SNOWFLAKE_SAMPLE_DATA.INFORMATION_SCHEMA.TABLES
    WHERE    table_schema = 'TPCH_SF1'
    ORDER BY table_name
""").to_pandas()

print('Tables in SNOWFLAKE_SAMPLE_DATA.TPCH_SF1:')
print(schema_df.to_string(index=False))

In [ ]:
SCHEMA_CONTEXT = """
Tables — always use the full name SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.<table>:
- SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER(C_CUSTKEY, C_NAME, C_ADDRESS, C_NATIONKEY, C_PHONE, C_ACCTBAL, C_MKTSEGMENT)
- SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS(O_ORDERKEY, O_CUSTKEY, O_ORDERSTATUS, O_TOTALPRICE, O_ORDERDATE, O_ORDERPRIORITY)
- SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.LINEITEM(L_ORDERKEY, L_PARTKEY, L_SUPPKEY, L_QUANTITY, L_EXTENDEDPRICE, L_DISCOUNT, L_TAX, L_SHIPDATE)
- SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.NATION(N_NATIONKEY, N_NAME, N_REGIONKEY)
- SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.REGION(R_REGIONKEY, R_NAME)
- SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.SUPPLIER(S_SUPPKEY, S_NAME, S_NATIONKEY, S_ACCTBAL)
"""


def ask_structured_data(question: str, show_sql: bool = True) -> pd.DataFrame:
    prompt = (
        'You are a Snowflake SQL expert.\n'
        'Write a single valid Snowflake SQL SELECT query that answers the question.\n'
        'Use only the tables in the schema. Return ONLY the SQL — no explanation, no markdown.\n\n'
        f'Schema:\n{SCHEMA_CONTEXT}\n\n'
        f'Question: {question}\nSQL:'
    )
    sql = Complete('mistral-large2', prompt).strip()
    sql = '\n'.join(l for l in sql.splitlines() if not l.strip().startswith('```')).strip()
    sql = sql.rstrip(';').strip()
    if show_sql:
        print(f'Generated SQL:\n{sql}\n')
    return session.sql(sql).limit(10).to_pandas()


# First question
print(ask_structured_data('Who are the top 5 customers by total order value?').to_string(index=False))

In [ ]:
more_questions = [
    'How many orders were placed in each year?',
    'Which nation has the highest average customer account balance?',
    'What are the top 3 order priorities by total revenue?',
]

for q in more_questions:
    print(f'\n{"="*60}\nQ: {q}')
    print(ask_structured_data(q).to_string(index=False))

---
## Conclusion

### What You Built
- **Module 1:** Called LLMs with `CORTEX.COMPLETE`, demonstrated hallucination, and applied
  RAG grounding with live data from `SNOWFLAKE_SAMPLE_DATA.TPCH_SF1`
- **Module 2:** Production RAG pipeline — chunked text, `CORTEX SEARCH SERVICE`,
  `RAG_App` class with hybrid retrieval and grounded generation
- **Module 3:** Text-to-SQL function that translates plain-English questions into
  executable Snowflake queries over pre-loaded sample data

### Next Steps
| Idea | Resource |
|------|----------|
| Upload real PDFs with `PARSE_DOCUMENT` | [Cortex Search docs](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview) |
| Add a chat UI with Streamlit in Snowflake | [Streamlit docs](https://docs.snowflake.com/en/developer-guide/streamlit/about-streamlit) |
| Production Text-to-SQL with a semantic model | [Cortex Analyst](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst) |
| Combine RAG + SQL in one agent | [Cortex Agents Quickstart](https://quickstarts.snowflake.com/guide/getting_started_with_cortex_agents) |